In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Investment Factor 계산에 필요한 데이터를 input 폴더에서 불러옴
adj_close       = pd.read_csv("./input/수정주가_현금배당반영.csv", index_col=0, parse_dates=True)
close           = pd.read_csv("./input/종가.csv", index_col=0, parse_dates=True)
share           = pd.read_csv("./input/상장주식수_보통주.csv", index_col=0, parse_dates=True)
mkt_ret         = pd.read_csv("./output/mkt_portfolio_ret.csv", index_col=0, parse_dates=True).squeeze()

---
#### **Pre-ranking beta 계산**

In [10]:
ME              = close * share
monthly_returns = adj_close.pct_change(fill_method=None)

In [11]:
june_ends = pd.to_datetime([f"{y}-06-30" for y in range(1999, 2025 + 1)])

tickers = monthly_returns.columns.tolist()

In [12]:
# date = '2017-06-30' # june_ends[-1]

# ticker = tickers[0]
# ticker = 'A950130'

# ret     = monthly_returns.loc[:date, ticker]
# mkt_t   = mkt_ret.loc[:date]
# mkt_t_1 = mkt_t.shift(1)

# df = pd.concat([
#     ret.rename('ret'),
#     mkt_t.rename('mkt_t'),
#     mkt_t_1.rename('mkt_t_1')
# ], axis=1).dropna().iloc[-60:]

# if len(df) < 24:
#     beta_sum = np.nan
# else:
#     X = sm.add_constant(df[['mkt_t', 'mkt_t_1']])
#     res = sm.OLS(df['ret'], X).fit()
#     beta_sum = res.params['mkt_t'] + res.params['mkt_t_1']

# beta_sum, len(df)

In [13]:
from joblib import Parallel, delayed

def calc_beta_sum(date, ticker):
    ret = monthly_returns.loc[:date, ticker]
    mkt_t = mkt_ret.loc[:date]
    mkt_t_1 = mkt_t.shift(1)

    df = pd.concat([
        ret.rename('ret'),
        mkt_t.rename('mkt_t'),
        mkt_t_1.rename('mkt_t_1')
    ], axis=1).iloc[-60:].dropna()

    if len(df) < 24:
        return date, ticker, np.nan

    X = sm.add_constant(df[['mkt_t', 'mkt_t_1']])
    res = sm.OLS(df['ret'], X).fit()
    beta_sum = res.params['mkt_t'] + res.params['mkt_t_1']
    return date, ticker, beta_sum

results = Parallel(n_jobs=-1, backend='loky')(
    delayed(calc_beta_sum)(date, ticker)
    for date in june_ends
    for ticker in tickers
)

beta_sum_df = (
    pd.DataFrame(results, columns=['Date', 'Ticker', 'beta_sum'])
    .pivot(index='Date', columns='Ticker', values='beta_sum')
    .sort_index()
)

In [14]:
pre_ranking_beta = beta_sum_df.copy()

In [15]:
mask_1 = ME.loc[pre_ranking_beta.index]
mask_2 = ME.shift(6).loc[pre_ranking_beta.index]


pre_ranking_beta = pre_ranking_beta.where(
    ME.loc[pre_ranking_beta.index].notna() &
    ME.shift(6).loc[pre_ranking_beta.index].notna()
)

In [16]:
pre_ranking_beta.to_csv("./output/pre_ranking_beta.csv")